In [1]:
import spacy
nlp = spacy.load('de_core_news_md')
import pandas as pd
import os

In [2]:

def parse_text_to_df(input_text):
    lines = input_text.splitlines()
    df = pd.DataFrame(columns=['Chapter', 'These', 'Text'])
    chapter = None
    these = ''
    text = ''

    for line in lines:
        if line.startswith("### "):  # Kapitel

            if chapter != line[4:].strip() and chapter != None:
                df.loc[len(df)] = [chapter, these, text.strip()]
                text = ''

            chapter = line[4:].strip()
            these = None #Rücksetzten der These bei neuem Kapitel

        elif line.startswith("## "):  #These

            if these != line[3:].strip():
                df.loc[len(df)] = [chapter, these, text.strip()]
                text = ''

            these = line[3:].strip()
        else:  # Kombinieren aufainanderfolgender Texte im Unterkapitel
            text += line[1:].strip()


    return df


In [3]:
file_name = 'all_parties_text_combined.parquet' 

if os.path.exists(file_name) == False:
    parties_file_list = ['spd.txt', 'cdu.txt', 'gruene.txt', 'afd.txt', 'fdp.txt', 'linke.txt']
    all_textes_df = pd.DataFrame()
    for party_file in parties_file_list:
        with open (party_file, 'r', encoding="utf-8") as f:
            text = f.read()

        df = parse_text_to_df(text)
        df['Party'] = party_file[:-4]

        all_textes_df = pd.concat([all_textes_df, df], ignore_index=True)

    all_textes_df = all_textes_df[['Party', 'Chapter', 'These', 'Text']]
    
    all_textes_df.to_parquet(file_name, index=False)

In [4]:
all_textes_df = pd.read_parquet("all_parties_text_combined.parquet")
all_textes_df.nunique()

Party        6
Chapter     61
These      562
Text       607
dtype: int64

In [5]:
all_textes_df['Party'].value_counts()

Party
gruene    145
fdp       124
spd        99
cdu        90
linke      85
afd        82
Name: count, dtype: int64